# 02 - Monitoring

Paste each Bash block into an Ubuntu terminal in order. Keep one terminal for this notebook. Complete notebook 01 first. Start monitoring before storage, databases or compute so their first startup logs and metrics are collected. All collector settings and scrape targets are supplied now; no later parameter repair is required. This installs Prometheus, Loki/Alloy, Grafana and Jaeger.


## Open your workspace

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Storage remains on PVCs: data-prometheus-0 (5 GiB, /prometheus), data-loki-0 (5 GiB, /loki), data-alloy-0 (1 GiB, /var/lib/alloy), data-grafana-0 (2 GiB, /var/lib/grafana) and data-jaeger-0 (5 GiB, /data). Use kubectl get pvc,pv to see generated PV names. All endpoints are ClusterIP; browser access is through loopback forwards inside the Guacamole Ubuntu desktop. Logs are collected from bigdata and headlamp; host applications outside Kubernetes are not part of this collector.

## Apply common telemetry configuration

```bash
kubectl apply -f monitoring/telemetry-config.yaml
```



## Apply loki

```bash
kubectl apply -f monitoring/loki.yaml
```



## Wait for loki

```bash
kubectl rollout status statefulset/loki --timeout=600s
```



## Apply alloy

```bash
kubectl apply -f monitoring/alloy.yaml
```



## Wait for alloy

```bash
kubectl rollout status statefulset/alloy --timeout=600s
```



## Apply prometheus

```bash
kubectl apply -f monitoring/prometheus.yaml
```



## Wait for prometheus

```bash
kubectl rollout status statefulset/prometheus --timeout=600s
```



## Apply jaeger

```bash
kubectl apply -f monitoring/jaeger.yaml
```



## Wait for jaeger

```bash
kubectl rollout status statefulset/jaeger --timeout=600s
```



## Apply grafana

```bash
kubectl apply -f monitoring/grafana.yaml
```



## Wait for grafana

```bash
kubectl rollout status statefulset/grafana --timeout=600s
```



## Inspect monitoring storage

```bash
kubectl get pods -l app=loki
kubectl get pvc
kubectl get services
```



## Check monitoring APIs and a real trace

```bash
bash validations/verify-monitoring.sh
```

Checks Prometheus, Grafana provisioning and a trace round-trip through Jaeger.

## Check collected logs and metrics

```bash
python3 validations/check-observability.py --components loki,alloy,prometheus,jaeger,grafana --jobs prometheus,grafana,loki,alloy
```

The helper makes temporary loopback API forwards, waits for actual log/metric samples and closes its forwards. It fails if collection is missing. Allow several scrape intervals after startup.

## Check Headlamp collection

```bash
python3 validations/check-observability.py --components headlamp --namespace headlamp
```

Headlamp logs use its standard application label; its namespace is included in the collector and performance dashboard.

## Open monitoring UIs

Run this in a second terminal and keep it open. Continue the setup in your original terminal.

```bash
bash monitoring-access.sh
```

Keep this second terminal open. In the Guacamole desktop browser: Grafana http://127.0.0.1:3001 (team / team1234), Prometheus http://127.0.0.1:9090 and Jaeger http://127.0.0.1:16686. No laptop tunnel is needed.

In Grafana open Dashboards, Training workload performance for CPU/memory, or Training centralized logs for collected logs. In Explore select Loki and enter `{namespace="bigdata"}`. For one service use `{component="postgres"}`. Prometheus targets for components not started yet, or deliberately paused, appear down; each later notebook verifies the active component targets. JVM panels populate after YARN, Hive and Kafka start. Jaeger has traces only when a workload explicitly emits them.

## Stop monitoring

```bash
bash stop-monitoring.sh
```

All monitoring PVCs are retained. Log collection pauses while Alloy is stopped.

## Resume monitoring

```bash
bash start-monitoring.sh
```



## Check collection after resuming

```bash
python3 validations/check-observability.py --components loki,alloy,prometheus,grafana --jobs prometheus,grafana,loki,alloy
```

